# Hierarchical optimization checks

Interactive equivalents of `test_hierarchical_optimize.py`. A deterministic in-notebook optimizer makes pruning and provenance easy to inspect without running optional quantum backends.

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys
from rdkit import Chem

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'src').is_dir():
    project_root = project_root.parent
if not (project_root / 'src' / 'ensemblelab').is_dir():
    raise RuntimeError('Open this notebook from within the ensemblelab repository.')
sys.path.insert(0, str(project_root / 'src'))

from ensemblelab import Ensemble, generate
from ensemblelab.generators import Conformer
from ensemblelab.optimizers import BaseOptimizer


ImportError: cannot import name 'hierarchical_optimize' from 'ensemblelab' (C:\Users\avirn\Dhruvcoding\ase\src\ensemblelab\__init__.py)

## Deterministic optimizer for workflow behavior

This lightweight implementation assigns known energies. It isolates hierarchical sorting and pruning from any force-field-specific behavior.

In [2]:
class EnergyOptimizer(BaseOptimizer):
    def __init__(self, method: str, energies: dict[int, float]) -> None:
        self._method = method
        self._energies = energies
        self.input_sizes: list[int] = []

    @property
    def method(self) -> str:
        return self._method

    def optimize(self, ensemble: Ensemble) -> Ensemble:
        self.input_sizes.append(len(ensemble.conformers))
        conformers = tuple(
            Conformer(
                id=conformer.id,
                atoms=conformer.atoms.copy(),
                energy=self._energies[conformer.id],
                energy_unit='kcal/mol',
                optimization_method=self.method,
                optimization_converged=True,
            )
            for conformer in ensemble.conformers
        )
        return Ensemble(
            smiles=ensemble.smiles,
            molecule=Chem.Mol(ensemble.molecule),
            conformers=conformers,
            metadata=deepcopy(ensemble.metadata),
        )

    def _optimize_conformer(self, molecule, conformer_id, atoms):
        raise AssertionError('This notebook optimizer overrides optimize directly.')

    def _history_settings(self) -> dict[str, object]:
        return {}


## Two-stage workflow

The first stage retains `max(target_size, ceil(5 * 0.4)) = 2` lowest energy conformers. The final stage optimizes only those two, then records complete stage provenance.

In [3]:
ensemble = generate('CCCC', n_confs=5)
ids = ensemble.conformer_ids
first = EnergyOptimizer('first', dict(zip(ids, [4.0, 1.0, 3.0, 2.0, 5.0], strict=True)))
final = EnergyOptimizer('final', dict(zip(ids, [0.4, 0.1, 0.3, 0.2, 0.5], strict=True)))

optimized = hierarchical_optimize(
    ensemble, workflow=[first, final], target_size=2, retention_rates=[0.4]
)

assert first.input_sizes == [5]
assert final.input_sizes == [2]
assert optimized.conformer_ids == (ids[1], ids[3])
assert [conformer.energy for conformer in optimized.conformers] == [0.1, 0.2]
assert all(conformer.energy is None for conformer in ensemble.conformers)

print('Hierarchical pruning contract passed.')
optimized.metadata['hierarchical_optimization']


Hierarchical pruning contract passed.


{'workflow': ['first', 'final'],
 'target_size': 2,
 'retention_rates': [0.4],
 'stages': [{'optimizer': 'first',
   'input_size': 5,
   'output_size': 2,
   'minimum_energy': 1.0,
   'maximum_energy': 5.0,
   'retained_energy_cutoff': 2.0,
   'energy_unit': 'kcal/mol'},
  {'optimizer': 'final',
   'input_size': 2,
   'output_size': 2,
   'minimum_energy': 0.1,
   'maximum_energy': 0.2,
   'retained_energy_cutoff': 0.2,
   'energy_unit': 'kcal/mol'}]}

## Workflow validation

A workflow is limited to three optimizer instances, and every non-final stage needs one retention rate.

In [4]:
validation_ensemble = generate('CCO', n_confs=1)
optimizer = EnergyOptimizer('test', {validation_ensemble.conformer_ids[0]: 0.0})

try:
    hierarchical_optimize(
        validation_ensemble,
        workflow=[optimizer, optimizer, optimizer, optimizer],
        retention_rates=[0.5, 0.5, 0.5],
    )
except ValueError as error:
    assert 'at most three' in str(error)
    print(f'Expected stage limit error: {error}')

try:
    hierarchical_optimize(validation_ensemble, workflow=[optimizer, optimizer])
except ValueError as error:
    assert 'retention_rates' in str(error)
    print(f'Expected retention-rate error: {error}')


Expected stage limit error: workflow may contain at most three optimizer instances.
Expected retention-rate error: retention_rates must have one value per non-final workflow stage.


## hierarchical_optimize() working example

In [17]:
from ensemblelab.optimizers.mmff import MMFFOptimizer
from ensemblelab.optimizers.xtb import GFN2xTBOptimizer
ensemble = generate('CCO', n_confs=20)

hierarchical_optimized = hierarchical_optimize(
    ensemble,
    workflow=[MMFFOptimizer(), MMFFOptimizer()],
    target_size=5,
    retention_rates=[0.5])


print(f"""
Hierarchical optimization completed with {len(hierarchical_optimized.conformers)} conformers remaining.
{hierarchical_optimized.metadata['hierarchical_optimization']}

Minimum energy conformer: {round(hierarchical_optimized.metadata["hierarchical_optimization"]["stages"][-1]["minimum_energy"], 2)} kcal/mol
""")


Hierarchical optimization completed with 5 conformers remaining.
{'workflow': ['MMFF', 'MMFF'], 'target_size': 5, 'retention_rates': [0.5], 'stages': [{'optimizer': 'MMFF', 'input_size': 20, 'output_size': 10, 'minimum_energy': -1.517097576376035, 'maximum_energy': -1.3368570615014994, 'retained_energy_cutoff': -1.3368570636825323, 'energy_unit': 'kcal/mol'}, {'optimizer': 'MMFF', 'input_size': 10, 'output_size': 5, 'minimum_energy': -1.517097576712211, 'maximum_energy': -1.336857063791254, 'retained_energy_cutoff': -1.3368570639673605, 'energy_unit': 'kcal/mol'}]}

Minimum energy conformer: -1.52 kcal/mol

